# 04 — Format Features: Per-Feature Breakdown
## Genna Content Machine — Format Analysis

Para cada uma das 15 format features, uma tabela mostrando **métricas medianas** e **taxas de engajamento** por categoria.

- Colunas de **taxa** (gradiente vermelho→verde): Like Rate, Comment Rate, Share Rate, Save Rate, Interaction Rate
- Colunas de **volume** (gradiente azul): Reach, Likes, Comments, Shares, Saves, Interactions
- Taxas = métrica / reach (mediana por post em cada categoria)
- Linha **OVERALL** = baseline geral para comparação

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, HTML

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'format_dataset.parquet')
with open(PROJECT_ROOT / 'format_features_reference.json', 'r', encoding='utf-8') as f:
    ref = json.load(f)

features = ref['features']
print(f'Dataset: {df.shape[0]} posts x {df.shape[1]} colunas')
print(f'Format features: {len(features)}')

In [ ]:
SCORE_FEATURES = {
    'plot_twist_strength', 'relatability_score', 'educational_entertainment_balance',
    'hook_strength', 'call_to_action_quality', 'cultural_relevance',
    'audience_familiarity_with_subject', 'presenter_energy_level', 'visual_relevance',
}

MIN_N = 2

if 'interactions' not in df.columns:
    interaction_cols = ['likes', 'comments', 'shares', 'saved']
    avail = [c for c in interaction_cols if c in df.columns]
    df['interactions'] = df[avail].apply(pd.to_numeric, errors='coerce').sum(axis=1)


def build_table(feature, data):
    sub = data[data[feature].notna() & data['reach'].notna() & (data['reach'] > 0)].copy()
    if len(sub) < 3:
        return None

    if feature in SCORE_FEATURES:
        sub['_grp'] = pd.to_numeric(sub[feature], errors='coerce').astype('Int64').astype(str)
    else:
        sub['_grp'] = sub[feature].astype(str)

    sub = sub.dropna(subset=['_grp'])

    for m in ['likes', 'comments', 'shares', 'saved', 'interactions']:
        if m in sub.columns:
            sub[f'_r_{m}'] = pd.to_numeric(sub[m], errors='coerce') / pd.to_numeric(sub['reach'], errors='coerce') * 100

    def agg(g):
        row = {'n': len(g), 'Reach': pd.to_numeric(g['reach'], errors='coerce').median()}
        for m, label in [('likes', 'Likes'), ('comments', 'Comments'), ('shares', 'Shares'), ('saved', 'Saves'), ('interactions', 'Interactions')]:
            if m in g.columns:
                row[label] = pd.to_numeric(g[m], errors='coerce').median()
            rate_col = f'_r_{m}'
            rate_label = f'{label.rstrip("s") if label != "Saves" else "Save"} Rate' if label != 'Interactions' else 'Interaction Rate'
            if label == 'Likes': rate_label = 'Like Rate'
            if label == 'Comments': rate_label = 'Comment Rate'
            if label == 'Shares': rate_label = 'Share Rate'
            if label == 'Saves': rate_label = 'Save Rate'
            if label == 'Interactions': rate_label = 'Interaction Rate'
            if rate_col in g.columns:
                row[rate_label] = g[rate_col].median()
        return row

    rows = {}
    for cat, g in sub.groupby('_grp', observed=True):
        if len(g) < MIN_N:
            continue
        rows[str(cat)] = agg(g)

    if not rows:
        return None

    rows['OVERALL'] = agg(sub)

    result = pd.DataFrame.from_dict(rows, orient='index')
    result.index.name = feature

    overall = result.loc[['OVERALL']]
    rest = result.drop('OVERALL')

    if feature in SCORE_FEATURES:
        rest = rest.loc[sorted(rest.index, key=lambda x: int(float(x)))]
    else:
        rest = rest.sort_values('Reach', ascending=False)

    return pd.concat([rest, overall])


def style_table(t):
    rate_cols = [c for c in t.columns if 'Rate' in c]
    abs_cols = [c for c in ['Reach', 'Likes', 'Comments', 'Shares', 'Saves', 'Interactions'] if c in t.columns]

    fmt = {'n': '{:.0f}', 'Reach': '{:,.0f}', 'Likes': '{:,.0f}', 'Comments': '{:,.0f}',
           'Shares': '{:,.0f}', 'Saves': '{:,.0f}', 'Interactions': '{:,.0f}'}
    for c in rate_cols:
        fmt[c] = '{:.2f}%'

    styled = (
        t.style
        .background_gradient(cmap='RdYlGn', subset=rate_cols)
        .background_gradient(cmap='Blues', subset=abs_cols)
        .format(fmt)
    )

    if 'OVERALL' in t.index:
        styled = styled.set_properties(
            subset=pd.IndexSlice['OVERALL', :],
            **{'font-weight': 'bold', 'border-top': '2px solid #333'}
        )

    return styled


print('Helpers loaded.')

In [ ]:
for feat_def in features:
    name = feat_def['name']
    ftype = feat_def.get('type', 'categorical')
    desc = feat_def.get('description', '')

    if name not in df.columns:
        continue

    display(HTML(f"<h3 style='margin-top:28px'>{name}</h3>"))
    display(HTML(
        f"<div style='margin:8px 0 12px 0;padding:10px 14px;background:#f8f9fa;"
        f"border-left:3px solid #2c3e50;border-radius:4px;font-size:13px'>"
        f"<b>Type:</b> {ftype} &nbsp;|&nbsp; <b>Description:</b> {desc}</div>"
    ))

    t = build_table(name, df)
    if t is not None:
        display(style_table(t))
    else:
        display(HTML("<p style='color:#999'>Not enough data for table.</p>"))